# Проект 1. 

**Болотин Платон · БАЦРФ 262**

**Бизнес-вопрос:** Как менялись успешные платежи клиентов, обращавшихся по вопросам оплаты, с апреля по сентябрь 2025 года и какие направления стоит проверить для увеличения их суммы?

**Основные гипотезы**:
1) Основной вклад в рост суммы успешных платежей между апрелем и сентябрём внесло увеличение числа платящих подписок.
2) Доля неуспешных попыток оплаты различается между тарифами.

# Данные и методика

В проекте рассматриваются клиенты, которые с 1 апреля по 30 сентября 2025 года обращались в поддержку по категории «оплата». Для них загружены платежи по подпискам, которыми они владеют, за тот же период. Правая граница 1 октября не включается. Выборка сформирована по обращениям за всё полугодие.
Одна строка исходной таблицы - это один платёж с идентификатором 'payment_id'. Подписки соединяются с клиентами через владельца, чтобы участники совместной подписки не размножали платежи. Для суммы успешных платежей учитывается только статус 'success'.

В первой гипотезе показатели рассчитываются на уровне «тариф × месяц». Это позволяет отдельно оценить вклад числа платящих подписок, количества платежей на подписку и среднего платежа внутри каждого тарифа. Апрель и сентябрь выбраны как начало и конец периода. Августовский пик рассматривается дополнительно, так как там самая большая сумма.

Во второй гипотезе доля неуспешных попыток считается на уровне платежных записей: 'failed / (success + failed)'. При сравнении успеха за 7, 14 и 30 дней учитываются только подписки с полным соответствующим сроком наблюдения. Канал привлечения берётся у владельца подписки, пропуски обозначаются как «Не указан».

In [ ]:
import os
import site
import stat
import sys
from pathlib import Path

for folder in site.getsitepackages():
    if sys.platform == "darwin":
        for path in Path(folder).glob("*prime*.pth"):
            flags = path.stat().st_flags
            if flags & stat.UF_HIDDEN:
                os.chflags(path, flags & ~stat.UF_HIDDEN)

    site.addsitedir(folder)

from prime.config import get_engine
print("Импорт работает")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sqlalchemy import text
import pandas as pd

engine = get_engine()

def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # День события считаем по UTC — так он у всех одинаковый.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        # Сервер общий на всю группу, соединения не копим.
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

# Загрузка данных

In [ ]:
mine_clients = q(f'''
select distinct client_id from prime.support_tickets
where category = 'оплата'
    and created_at >= '2025-04-01'
    and created_at <  '2025-10-01'
''')
print(mine_clients['client_id'].nunique())

In [ ]:
mine_clients = f'''
select distinct client_id from prime.support_tickets
where category = 'оплата'
    and created_at >= '2025-04-01'
    and created_at <  '2025-10-01'
'''

clients = q(f'''
    SELECT distinct client_id, registered_at, acquisition_channel, region_code
    FROM prime.clients
    WHERE client_id IN ({mine_clients})
    ORDER BY client_id
''')
print(clients['client_id'].nunique())
clients.head()

In [ ]:
df = q(f'''
    select
        payment_id,
        owner_client_id AS client_id,
        p.subscription_id,
        tariff_id,
        paid_at,
        amount,
        status,
        acquisition_channel,
        region_code
    from prime.payments p
    join prime.subscriptions s on p.subscription_id = s.subscription_id
    left join prime.clients c on s.owner_client_id = c.client_id
    WHERE s.owner_client_id IN ({mine_clients})
      AND p.paid_at >= '2025-04-01'
      AND p.paid_at < '2025-10-01'
    ORDER BY p.paid_at, p.payment_id
''')

print(len(df))
df.head()

In [ ]:
df['acquisition_channel'] = df['acquisition_channel'].fillna('Не указан')

In [ ]:
df = df.drop_duplicates()
print(len(df))

In [ ]:
df['client_id'].nunique()

In [ ]:
df['paid_at'] = pd.to_datetime(df['paid_at'], utc=True)
df['status'].value_counts()

In [ ]:
print('кол-во успешных платежей:', len(df[df['status'] == 'success'].copy()))
print('кол-во уникальных клиентов с успешным платежом:', df[df['status'] == 'success']['client_id'].nunique())
print('сумма по успемшным платежам:', round(float(df[df['status'] == 'success']['amount'].sum()), 2))

# Аналитика

In [ ]:
df_sucsess = df[df['status'] == 'success']
channel = df_sucsess.groupby('acquisition_channel')['amount'].sum().reset_index()
channel_plot = channel.sort_values('amount')

plt.figure(figsize=(10, 5))
plt.barh(
    channel_plot['acquisition_channel'],
    channel_plot['amount'],
    color='steelblue'
)
plt.title('Сумма платежей по каналам привлечения')
plt.xlabel('Сумма')
plt.ylabel('Канал привлечения')
plt.show()

In [ ]:
not_successful_pay = df[df['status'] != 'success']
channel_2 = not_successful_pay.groupby(['acquisition_channel', 'status'])['amount'].sum().reset_index()
channel_status = channel_2.pivot(
    index='acquisition_channel',
    columns='status',
    values='amount'
).fillna(0)

ax = channel_status.plot(
    kind='bar',
    figsize=(12, 6),
    rot=30
)

ax.set_title('Сумма по каналам привлечения и статусам платежей')
ax.set_xlabel('Канал привлечения')
ax.set_ylabel('Сумма')
ax.legend(title='Статус')
plt.show()

In [ ]:
status = df.groupby('status')['amount'].sum().reset_index()
print(status.head())
plt.figure(figsize=(8, 5))
plt.bar(
    status['status'],
    status['amount'],
    color='steelblue'
)
plt.title('Сумма по статусам платежей')
plt.xlabel('Статус')
plt.ylabel('Сумма')
plt.show()

In [ ]:
monthly_success = df[df['status'] == 'success']
monthly_success['month'] = pd.to_datetime(monthly_success['paid_at'], utc=True).dt.strftime('%Y-%m')
monthly_success_revenue = monthly_success.groupby('month')['amount'].sum().sort_index()

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(
    monthly_success_revenue.index,
    monthly_success_revenue.values,
    color='steelblue'
)

ax.set_ylim(
    monthly_success_revenue.min() * 0.97,
    monthly_success_revenue.max() * 1.03
)

ax.bar_label(
    bars,
    labels=[
        f'{amount:,.0f}'.replace(',', ' ')
        for amount in monthly_success_revenue.values
    ],
    padding=5
)

ax.set_title(
    'Сумма успешных платежей по месяцам'
)
ax.set_xlabel('Месяц')
ax.set_ylabel('Сумма')
ax.grid(axis='y', alpha=0.2)
plt.show()

In [ ]:
not_successful_pay = df[df['status'] != 'success']
not_successful_pay['month'] = pd.to_datetime(not_successful_pay['paid_at'], utc=True).dt.strftime('%Y-%m')
monthly_revenue = not_successful_pay.groupby(['month', 'status'])['amount'].sum().sort_index().reset_index()

month_status = monthly_revenue.pivot(
    index='month',
    columns='status',
    values='amount'
).fillna(0)

ax = month_status.plot(
    kind='bar',
    figsize=(12, 6),
    rot=30
)

ax.set_title('Сумма по месяцам и статусам платежей')
ax.set_xlabel('Месяц')
ax.set_ylabel('Сумма')
ax.legend(title='Статус')
plt.show()

In [ ]:
print('кол-во уникальных регионов:', df['region_code'].nunique())
region = df_sucsess.groupby('region_code')['amount'].sum().reset_index()
region_plot = region.nlargest(10, 'amount').sort_values('amount')
region_plot['region_code'] = region_plot['region_code'].astype(str)

plt.figure(figsize=(10, 7))

plt.barh(
    region_plot['region_code'],
    region_plot['amount'],
    color='steelblue'
)

plt.title('Топ 10 регионов по сумме платежей')
plt.xlabel('Сумма')
plt.ylabel('Код региона')
plt.show()

# Гипотеза №1

**Гипотеза**: основной вклад в рост суммы успешных платежей между апрелем и сентябрём внесло увеличение числа платящих подписок.

In [ ]:
payments = df.copy()
payments['paid_at'] = pd.to_datetime(payments['paid_at'], utc=True)
payments['amount'] = pd.to_numeric(payments['amount'], errors='raise')

period_start = pd.Timestamp('2025-04-01', tz='UTC')
period_end = pd.Timestamp('2025-10-01', tz='UTC')

base_month = '2025-04'
compare_month = '2025-09'

payments = payments[(payments['paid_at'] >= period_start) & (payments['paid_at'] < period_end)]
payments['month'] = payments['paid_at'].dt.strftime('%Y-%m')

successful_pay = payments[payments['status'] == 'success']

In [ ]:
calendar_days = pd.date_range(start=period_start, end=period_end, inclusive='left')

observed_days = pd.DatetimeIndex(payments['paid_at'].dt.normalize().unique())
missing_days = calendar_days.difference(observed_days)

print('Дней без строк с платежами:', len(missing_days))
missing_comparison_days = missing_days[missing_days.strftime('%Y-%m').isin([base_month, compare_month])]

In [ ]:
monthly = successful_pay.groupby(['month', 'tariff_id']).agg(
    revenue=('amount', 'sum'),
    payments=('payment_id', 'size'),
    paying_subscriptions=('subscription_id', 'nunique')
).reset_index()

# Среднее количество успешных платежей на платящую подписку.
monthly['payments_per_subscription'] = monthly['payments'] / monthly['paying_subscriptions']

# Средний успешный платёж.
monthly['average_payment'] = monthly['revenue'] / monthly['payments']
monthly = monthly.sort_values(['month', 'tariff_id'])

monthly.round(2)

In [ ]:
base = monthly[monthly['month'] == base_month]
compare = monthly[monthly['month'] == compare_month]

comparison = base.merge(
    compare, on='tariff_id',
    how='outer', suffixes=('_base', '_compare'),
    validate='1:1',
    indicator=True
)

comparison = comparison.drop(columns='_merge')

base_revenue = comparison['revenue_base'].sum()
compare_revenue = comparison['revenue_compare'].sum()

revenue_change = compare_revenue - base_revenue
revenue_change_pct = revenue_change / base_revenue * 100

print('Сумма успешных платежей в апреле:', round(base_revenue, 2))
print('Сумма успешных платежей в сентябре:', round(compare_revenue, 2))
print('Изменение в рублях:', round(revenue_change, 2))
print('Изменение в процентах:', round(revenue_change_pct, 2))

In [ ]:
base_subscriptions = comparison['paying_subscriptions_base']
base_frequency = comparison['payments_per_subscription_base']
base_average_payment = comparison['average_payment_base']

compare_subscriptions = comparison['paying_subscriptions_compare']
compare_frequency = comparison['payments_per_subscription_compare']
compare_average_payment = comparison['average_payment_compare']

# Вклад изменения числа платящих подписок.
# Частоту и средний платёж пока оставляем апрельскими.
comparison['effect_subscriptions'] = (compare_subscriptions - base_subscriptions) * base_frequency * base_average_payment

# Вклад изменения частоты платежей.
# Число подписок уже сентябрьское, средний платёж ещё апрельский.
comparison['effect_frequency'] = compare_subscriptions * (compare_frequency - base_frequency) * base_average_payment

# Вклад изменения среднего платежа.
# Число подписок и частота уже сентябрьские.
comparison['effect_average_payment'] = compare_subscriptions * compare_frequency * (compare_average_payment - base_average_payment)

comparison['revenue_change'] = comparison['revenue_compare'] - comparison['revenue_base']

# Складываем три вклада для каждого тарифа.
comparison['effects_total'] = comparison['effect_subscriptions'] + comparison['effect_frequency'] + comparison['effect_average_payment']

comparison[[
    'tariff_id',
    'revenue_base',
    'revenue_compare',
    'effect_subscriptions',
    'effect_frequency',
    'effect_average_payment',
    'revenue_change'
]].round(2)

In [ ]:
effects = pd.DataFrame({
    'factor': [
        'Число платящих подписок',
        'Платежей на платящую подписку',
        'Средний платёж'
    ],
    'effect': [
        comparison['effect_subscriptions'].sum(),
        comparison['effect_frequency'].sum(),
        comparison['effect_average_payment'].sum()
    ]
})

effects.round(2)

In [ ]:
colors = ['seagreen' if value >= 0 else 'indianred' for value in effects['effect']]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(
    [
        'Число платящих\nподписок',
        'Платежей на\nплатящую подписку',
        'Средний\nплатёж'
    ],
    effects['effect'],
    color=colors
)

labels = [
    f'{value:,.2f}'.replace(',', ' ')
    for value in effects['effect']
]

ax.bar_label(bars, labels=labels, padding=5)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Вклад факторов в изменение суммы: сентябрь к апрелю')
ax.set_ylabel('Вклад')
ax.margins(y=0.2)
plt.show()

## Из каких каналов пришли клиенты?

In [ ]:
channel_payments = df[df['status'] == 'success']
channel_payments['month'] = pd.to_datetime(channel_payments['paid_at'], utc=True).dt.strftime('%Y-%m')

channel_payments['acquisition_channel'] = channel_payments['acquisition_channel'].fillna('Не указан')

channels_per_subscription = channel_payments.groupby('subscription_id')['acquisition_channel'].nunique()

april_subscriptions = (
    channel_payments[channel_payments['month'] == '2025-04'][['subscription_id', 'acquisition_channel']].drop_duplicates('subscription_id')
)

september_subscriptions = (
    channel_payments[channel_payments['month'] == '2025-09'][['subscription_id', 'acquisition_channel']].drop_duplicates('subscription_id')
)

subscription_flow = april_subscriptions.merge(
    september_subscriptions,
    on=['subscription_id', 'acquisition_channel'],
    how='outer',
    indicator=True,
    validate='1:1'
)

subscription_flow['paid_both_months'] = subscription_flow['_merge'] == 'both'

subscription_flow['only_september'] = subscription_flow['_merge'] == 'right_only'

subscription_flow['only_april'] = subscription_flow['_merge'] == 'left_only'

channel_growth = subscription_flow.groupby('acquisition_channel').agg(
    paid_both_months=('paid_both_months', 'sum'),
    only_september=('only_september', 'sum'),
    only_april=('only_april', 'sum')
).reset_index()

channel_growth['april_subscriptions'] = channel_growth['paid_both_months'] + channel_growth['only_april']

channel_growth['september_subscriptions'] = channel_growth['paid_both_months'] + channel_growth['only_september']

channel_growth['net_growth'] = channel_growth['only_september'] - channel_growth['only_april']
channel_growth = channel_growth.sort_values('net_growth', ascending=False)

print('Платящих подписок в апреле:', len(april_subscriptions))
print('Платящих подписок в сентябре:', len(september_subscriptions))
print('Общий прирост:', channel_growth['net_growth'].sum())

In [ ]:
print('Платили в обоих месяцах:', int(subscription_flow['paid_both_months'].sum()))
print('Платили в сентябре, но не в апреле:', int(subscription_flow['only_september'].sum()))
print('Платили в апреле, но не в сентябре:', int(subscription_flow['only_april'].sum()))

In [ ]:
channel_growth_plot = channel_growth.sort_values('net_growth')

colors = [
    'seagreen' if value >= 0 else 'indianred'
    for value in channel_growth_plot['net_growth']
]

fig, ax = plt.subplots(figsize=(11, 6))

bars = ax.barh(
    channel_growth_plot['acquisition_channel'],
    channel_growth_plot['net_growth'],
    color=colors
)

ax.bar_label(
    bars,
    labels=[
        f'{value:+.0f}'
        for value in channel_growth_plot['net_growth']
    ],
    padding=5
)

ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Изменение числа платящих подписок: сентябрь к апрелю')
ax.set_xlabel('Прирост или снижение, подписок')
ax.set_ylabel('Канал привлечения владельца')
ax.margins(x=0.2)
plt.show()

## Почему максимальная сумма была в августе, а потом в сентябре она упала?

In [ ]:
peak_monthly = monthly[monthly['month'].isin(['2025-07', '2025-08', '2025-09'])]
peak_monthly['days'] = pd.to_datetime(peak_monthly['month']).dt.days_in_month

In [ ]:
peak_totals = peak_monthly.groupby('month').agg(
    revenue=('revenue', 'sum'),
    days=('days', 'first')
).reset_index()

peak_totals['revenue_per_day'] = (
    peak_totals['revenue'] / peak_totals['days']
)

display(peak_totals.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, column, title in zip(
    axes,
    ['revenue', 'revenue_per_day'],
    ['Сумма за календарный месяц', 'Средняя сумма за календарный день']
):
    ax.plot(
        peak_totals['month'],
        peak_totals[column],
        marker='o',
        color='steelblue',
        linewidth=2
    )

    for month, value in zip(peak_totals['month'], peak_totals[column]):
        ax.annotate(
            f'{value:,.0f}'.replace(',', ' '),
            xy=(month, value),
            xytext=(0, 10),
            textcoords='offset points',
            ha='center'
        )

    ax.set_title(title)
    ax.set_xlabel('Месяц')
    ax.set_ylabel('Руб.')
    ax.ticklabel_format(style='plain', axis='y')
    ax.margins(y=0.3)
    ax.grid(alpha=0.2)


plt.show()

In [ ]:
def explain_month_change(month_before, month_after):
    before = monthly[monthly['month'] == month_before].copy()
    after = monthly[monthly['month'] == month_after].copy()

    comparison = before.merge(
        after,
        on='tariff_id',
        how='outer',
        suffixes=('_before', '_after'),
        validate='1:1',
        indicator=True
    )
    subscriptions_before = comparison['paying_subscriptions_before']
    frequency_before = comparison['payments_per_subscription_before']
    average_payment_before = comparison['average_payment_before']

    subscriptions_after = comparison['paying_subscriptions_after']
    frequency_after = comparison['payments_per_subscription_after']
    average_payment_after = comparison['average_payment_after']

    comparison['effect_subscriptions'] = (subscriptions_after - subscriptions_before) * frequency_before * average_payment_before
    comparison['effect_frequency'] = subscriptions_after * (frequency_after - frequency_before) * average_payment_before
    comparison['effect_average_payment'] = subscriptions_after * frequency_after * (average_payment_after - average_payment_before)
    effect_columns = [
        'effect_subscriptions',
        'effect_frequency',
        'effect_average_payment'
    ]

    actual_change = comparison['revenue_after'] - comparison['revenue_before']
    result = pd.DataFrame({
        'factor': [
            'Число платящих подписок',
            'Платежей на подписку за месяц',
            'Средний платёж'
        ],
        'effect': comparison[effect_columns].sum().values
    })

    print(f'\nСравнение: {month_before} → {month_after}')
    print('Изменение суммы, руб.:', round(actual_change.sum(), 2))
    print(result.round(2))

    return result


july_august = explain_month_change('2025-07', '2025-08')
august_september = explain_month_change('2025-08', '2025-09')

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(16, 6),
    sharex=True,
    sharey=True
)

for ax, result, title in zip(
    axes,
    [july_august, august_september],
    ['Почему сумма изменилась в августе', 'Почему сумма изменилась в сентябре']
):
    colors = [
        'seagreen' if value >= 0 else 'indianred'
        for value in result['effect']
    ]

    bars = ax.barh(
        result['factor'],
        result['effect'],
        color=colors
    )

    ax.bar_label(
        bars,
        labels=[
            f'{value:+,.0f}'.replace(',', ' ')
            for value in result['effect']
        ],
        padding=5
    )

    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel('Вклад в изменение суммы')
    ax.margins(x=0.35)
    ax.grid(axis='x', alpha=0.2)
    ax.set_axisbelow(True)

axes[0].invert_yaxis()

plt.tight_layout()
plt.show()

## Вывод по гипотезе 1

**Гипотеза поддерживается выбранным факторным разложением:** основной вклад в рост суммы успешных платежей между апрелем и сентябрём внесло увеличение числа платящих подписок.

Сумма успешных платежей выросла с 1 136 853 до 1 227 735, то есть на 90 882 или 7,99%. Число подписок хотя бы с одной успешной оплатой увеличилось с 4 119 до 4 439, то есть на 320. Средний платёж внутри тарифов остался равным 199 и 399. Таким образом, рост платящей базы обеспечил увеличение суммы, а небольшое снижение частоты платежей частично его компенсировало.

**В каких каналах сосредоточен прирост**

Наибольший абсолютный прирост платящих подписок наблюдался у владельцев из каналов organic, performance и referral. Вместе они обеспечили 72,5% чистого прироста. При этом 3 192 подписки платили в обоих сравниваемых месяцах, 1 247 — в сентябре, но не в апреле, а 927 — в апреле, но не в сентябре. Разница между последними двумя группами составила 320 подписок. 

**Почему максимум пришёлся на август**

В августе сумма успешных платежей достигла 1 278 353 - максимума за рассматриваемое полугодие. По сравнению с июлем она выросла на 11 161, потому что:
1) увеличение числа платящих подписок внесло +7 584
2) увеличение числа платежей на подписку за месяц +3 576
3) средний платёж внутри тарифов не изменился.

В сентябре сумма снизилась на 50 618. Вклад уменьшения числа платящих подписок составил −20 750, а снижения числа платежей на подписку за месяц −29 867.

**Рекомендации**

В первую очередь стоит разобраться, почему 927 подписок, по которым платили в апреле, не получили оплату в сентябре. Нужно посмотреть, закончились ли подписки, были ли отмены или неуспешные попытки оплаты. От причины зависит действие: кому-то может помочь напоминание, кому-то — помощь с оплатой, а кто-то уже не пользуется сервисом.
Также стоит выяснить, сколько из 1 247 сентябрьских плательщиков без апрельской оплаты — новые подписки, а сколько вернулись после перерыва. Так мы поймём, за счёт чего пополняется платящая база.
Organic, performance и referral дали наибольший абсолютный прирост. Но увеличивать расходы на эти каналы только на основании прироста я бы не стал: сначала нужно сравнить стоимость привлечения и то, как долго привлечённые клиенты продолжают платить.

# Гипотеза №2

**Гипотеза**: доля неуспешных попыток оплаты различается между тарифами.

In [ ]:
payment_check = df.copy()
payment_check['paid_at'] = pd.to_datetime(payment_check['paid_at'], utc=True)
period_start = pd.Timestamp('2025-04-01', tz='UTC')
period_end = pd.Timestamp('2025-10-01', tz='UTC')

payment_check = payment_check[(payment_check['paid_at'] >= period_start) & (payment_check['paid_at'] < period_end)]

payment_check['month'] = payment_check['paid_at'].dt.strftime('%Y-%m')
attempts = payment_check[payment_check['status'].isin(['success', 'failed'])]
attempts['is_failed'] = attempts['status'] == 'failed'

In [ ]:
failure_by_tariff = attempts.groupby('tariff_id').agg(
    attempt_count=('payment_id', 'size'),
    failed_count=('is_failed', 'sum'),
    subscriptions_with_attempts=('subscription_id', 'nunique')
).reset_index()

failure_by_tariff['failed_share_pct'] = failure_by_tariff['failed_count'] / failure_by_tariff['attempt_count'] * 100
failed_subscriptions = attempts[attempts['is_failed']].groupby('tariff_id')['subscription_id'].nunique().reset_index(name='subscriptions_with_failed')
failure_by_tariff = failure_by_tariff.merge(
    failed_subscriptions,
    on='tariff_id',
    how='left',
    validate='1:1'
)

failure_by_tariff['subscriptions_with_failed'] = failure_by_tariff['subscriptions_with_failed'].fillna(0).astype(int)

print(failure_by_tariff.round(2))
difference = failure_by_tariff['failed_share_pct'].max() - failure_by_tariff['failed_share_pct'].min()

print('Разница между тарифами, процентных пунктов:', round(difference, 2))

In [ ]:
tariff_names = {
    1: 'Тариф 199 ₽',
    2: 'Тариф 399 ₽'
}

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    failure_by_tariff['tariff_id'].map(tariff_names),
    failure_by_tariff['failed_share_pct'],
    color='indianred'
)
ax.bar_label(
    bars,
    labels=[
        f'{value:.2f}%'
        for value in failure_by_tariff['failed_share_pct']
    ],
    padding=5
)

ax.set_title('Доля неуспешных попыток оплаты по тарифам')
ax.set_xlabel('Тариф')
ax.set_ylabel('Failed / (success + failed)')
ax.set_ylim(0, max(failure_by_tariff['failed_share_pct'].max() * 1.25, 1))
ax.grid(axis='y', alpha=0.2)
plt.show()

In [ ]:
failure_monthly = attempts.groupby(['month', 'tariff_id']).agg(
    attempt_count=('payment_id', 'size'),
    failed_count=('is_failed', 'sum')
).reset_index()

failure_monthly['failed_share_pct'] = failure_monthly['failed_count'] / failure_monthly['attempt_count'] * 100
failure_plot = failure_monthly.pivot(
    index='month',
    columns='tariff_id',
    values='failed_share_pct'
)


months = pd.period_range('2025-04', '2025-09', freq='M').astype(str)
failure_plot = failure_plot.reindex(months)
failure_plot = failure_plot.rename(columns=tariff_names)

ax = failure_plot.plot(
    figsize=(11, 5),
    marker='o',
    linewidth=2
)

ax.set_title('Доля неуспешных попыток оплаты по месяцам')
ax.set_xlabel('Месяц')
ax.set_ylabel('Failed / (success + failed)')
ax.set_ylim(bottom=0)
ax.legend(title='Тариф')
ax.grid(alpha=0.2)
plt.show()

## Когда по подписке появилась успешная оплата после неуспешной попытки?

In [ ]:
payment_details = payment_check.copy()
payment_details['acquisition_channel'] = payment_details['acquisition_channel'].fillna('Не указан')

first_failure = (
    payment_details[payment_details['status'] == 'failed']
    .sort_values(['paid_at', 'payment_id'])
    .drop_duplicates('subscription_id')
    [[
        'subscription_id',
        'tariff_id',
        'acquisition_channel',
        'paid_at'
    ]]
    .rename(columns={'paid_at': 'first_failed_at'})
)

success_dates = (
    payment_details[payment_details['status'] == 'success']
    [['subscription_id', 'paid_at']]
    .rename(columns={'paid_at': 'success_at'})
)

possible_successes = first_failure[['subscription_id', 'first_failed_at']].merge(
    success_dates,
    on='subscription_id',
    how='left',
    validate='1:m'
)

possible_successes = possible_successes[possible_successes['success_at']> possible_successes['first_failed_at']]
nearest_success = possible_successes.groupby('subscription_id', as_index=False)['success_at'].min()

failure_followup = first_failure.merge(
    nearest_success,
    on='subscription_id',
    how='left',
    validate='1:1'
)

failure_followup['days_to_success'] = (
    failure_followup['success_at'] - failure_followup['first_failed_at']).dt.total_seconds() / (24 * 60 * 60)

found_success = failure_followup['success_at'].notna()

print('Подписок с неуспешной попыткой:', len(failure_followup))
print('С последующей успешной оплатой до конца выгрузки:', int(found_success.sum()))
print('Без обнаруженной последующей успешной оплаты:', int((~found_success).sum()))
print('Медиана дней до успеха среди найденных оплат:', round(failure_followup.loc[found_success, 'days_to_success'].median(), 2))

In [ ]:
failure_followup['success_interval'] = pd.cut(
    failure_followup['days_to_success'],
    bins=[0, 7, 14, 30, 60, float('inf')],
    labels=[
        'До 7 дней включительно',
        'Более 7 — до 14 дней',
        'Более 14 — до 30 дней',
        'Более 30 — до 60 дней',
        'Более 60 дней'
    ]
)

interval_counts = failure_followup['success_interval'].value_counts(sort=False)


fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(
    interval_counts.index.astype(str),
    interval_counts.values,
    color='steelblue'
)

ax.bar_label(bars, padding=5)
ax.set_title('Когда обнаружена ближайшая успешная оплата после failed')
ax.set_xlabel('Количество подписок')
ax.set_ylabel('Время после первой неуспешной попытки')
ax.margins(x=0.15)
plt.show()

In [ ]:
channel_attempts = payment_details[payment_details['status'].isin(['success', 'failed'])]

channel_attempts['is_failed'] = channel_attempts['status'] == 'failed'

channel_failures = channel_attempts.groupby('acquisition_channel').agg(
    attempt_count=('payment_id', 'size'),
    failed_count=('is_failed', 'sum'),
    subscription_count=('subscription_id', 'nunique')
).reset_index()

channel_failures['failed_share_pct'] = channel_failures['failed_count'] / channel_failures['attempt_count'] * 100
channel_failures = channel_failures.sort_values('failed_share_pct')

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(
    channel_failures['acquisition_channel'],
    channel_failures['failed_share_pct'],
    color='indianred'
)

labels = [
    f'{share:.2f}% ({int(failed)} из {int(total)})'
    for share, failed, total in zip(
        channel_failures['failed_share_pct'],
        channel_failures['failed_count'],
        channel_failures['attempt_count']
    )
]

ax.bar_label(bars, labels=labels, padding=5)
ax.set_title('Доля неуспешных попыток оплаты по каналам')
ax.set_xlabel('Failed / (success + failed)')
ax.set_ylabel('Канал привлечения')
ax.set_xlim(0, max(channel_failures['failed_share_pct'].max() * 1.5, 1))
plt.show()

In [ ]:
followup_tables = []

for horizon in [7, 14, 30]:
    complete_followup = failure_followup[failure_followup['first_failed_at'] + pd.Timedelta(days=horizon) < period_end]

    complete_followup['has_success'] = (
        complete_followup['success_at'].notna()
        & (complete_followup['days_to_success'] <= horizon)
    )

    result = complete_followup.groupby('acquisition_channel').agg(
        eligible_subscriptions=('subscription_id', 'size'),
        with_success=('has_success', 'sum')
    ).reset_index()

    result['success_share_pct'] = result['with_success'] / result['eligible_subscriptions'] * 100
    result['horizon_days'] = horizon

    followup_tables.append(result)

channel_followup = pd.concat(followup_tables, ignore_index=True)

display(channel_followup[[
    'acquisition_channel',
    'horizon_days',
    'eligible_subscriptions',
    'with_success',
    'success_share_pct'
]].round(2))

In [ ]:
followup_30 = channel_followup[
    channel_followup['horizon_days'] == 30
].sort_values('success_share_pct')


fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.barh(
    followup_30['acquisition_channel'],
    followup_30['success_share_pct'],
    color='seagreen'
)

labels = [
    f'{share:.1f}% ({int(success)} из {int(total)})'
    for share, success, total in zip(
        followup_30['success_share_pct'],
        followup_30['with_success'],
        followup_30['eligible_subscriptions']
    )
]

ax.bar_label(bars, labels=labels, padding=5)
ax.set_title('Успешная оплата в течение 30 дней после failed по каналам')
ax.set_xlabel('Доля подписок с последующей успешной оплатой')
ax.set_ylabel('Канал привлечения')
ax.set_xlim(0, 125)
ax.set_xticks(range(0, 101, 20))
plt.show()

## Вывод по гипотезе 2

**В выборке доли неуспешных попыток различаются между тарифами, но устойчивое преимущество одного тарифа не установлено.** Доля рассчитывалась как число платежей со статусом `failed`, делённое на число платежей со статусами `success` и `failed`. Возвраты в этот показатель не включались.

| Показатель | Тариф 199 ₽ | Тариф 399 ₽ |
|---|---:|---:|
| Количество попыток | 17 182 | 10 435 |
| Неуспешные попытки | 572 | 315 |
| Доля неуспешных попыток | 3,33% | 3,02% |

Разница составила 0,31 процентнов. Однако в июне и июле доля неуспешных попыток была выше у тарифа 399 ₽, а в остальные месяцы — у тарифа 199 ₽. 

**Что происходило после неуспешной попытки**

Для каждой подписки рассматривалась первая неуспешная попытка внутри периода и ближайшая последующая успешная оплата той же подписки. Неуспешные попытки обнаружены у 817 подписок. У 665 из них найдена последующая успешная оплата до конца выгрузки, у 152 — не найдена. Может предположить, что они ушли в отток. Среди подписок с найденной последующей оплатой медианное время до неё составило 30 дней. У подписок с полным соответствующим сроком наблюдения успешных оплат в течение 7 и 14 дней не обнаружено. 

**Различия между каналами привлечения**

Среди известных каналов доля неуспешных попыток находилась в диапазоне от 2,91% у social до 3,52% у crm. Разница составила около 0,61 процентов. Среди подписок с полными 30 днями наблюдения доля последующей успешной оплаты составила от 37,78% у social — 34 из 90 подписок — до 53,62% у referral — 37 из 69 подписок.

**Рекомендации**

Менять условия тарифов по этим результатам я бы не стал. Разница в долях неуспешных попыток небольшая, а тариф с более высокой долей менялся от месяца к месяцу. Полезнее посмотреть причины отказов: например, связаны ли они с недостатком средств, ограничениями банка или техническими ошибками.
Отдельно стоит проверить, что происходит после неудачной оплаты. Следующий успешный платёж среди найденных случаев обычно появляется примерно через месяц. Нужно понять, были ли до него повторные попытки и не является ли этот успех уже оплатой следующего периода.
После этого можно выбрать конкретное улучшение — например, напоминание или повторную попытку оплаты — и проверить его на небольшой группе. Результат стоит сравнить с группой, для которой порядок оплаты не менялся.